<a href="https://colab.research.google.com/github/DhairyaDave08/Sentiment-Analysis/blob/main/Executive_files/03_Execution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Execution**

In [31]:
import re
import pickle
import urllib.request
import os

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [32]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")
NON_ALPHA_PATTERN = re.compile(r"[^a-zA-Z\s]")
MULTI_SPACE_PATTERN = re.compile(r"\s+")

STOP_WORDS = set(stopwords.words("english"))
stemmer = PorterStemmer()

def clean_text(text):
    text = text.lower()
    text = URL_PATTERN.sub("", text)
    text = MENTION_PATTERN.sub("", text)
    text = HASHTAG_PATTERN.sub(r"\1", text)
    text = NON_ALPHA_PATTERN.sub(" ", text)
    text = MULTI_SPACE_PATTERN.sub(" ", text).strip()
    return text

def process(text):
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS]
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

In [33]:
os.makedirs("models", exist_ok=True)
model_url = "https://raw.githubusercontent.com/DhairyaDave08/Sentiment-Analysis/main/Executive_files/sentiment_model.pkl"
vectorizer_url = "https://raw.githubusercontent.com/DhairyaDave08/Sentiment-Analysis/main/Executive_files/vectorizer.pkl"

urllib.request.urlretrieve(model_url, "models/sentiment_model.pkl")
urllib.request.urlretrieve(vectorizer_url, "models/vectorizer.pkl")

with open("models/sentiment_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("models/vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

print("Model and vectorizer loaded successfully.")

Model and vectorizer loaded successfully.


In [36]:
import re
import numpy as np
from scipy.sparse import hstack, csr_matrix

NEGATION_WORDS = {
    "not", "no", "never", "cant", "cannot", "wont", "dont",
    "isnt", "arent", "wasnt", "werent", "didnt", "doesnt",
    "hasnt", "havent", "wouldnt", "shouldnt", "couldnt",
}

def negation_count(cleaned_text):
    tokens = cleaned_text.lower().split()
    return sum(1 for t in tokens if t in NEGATION_WORDS)

def exclamation_count(raw_text):
    return raw_text.count("!")

def question_count(raw_text):
    return raw_text.count("?")

def caps_ratio(raw_text):
    letters = [c for c in raw_text if c.isalpha()]
    if not letters:
        return 0.0
    caps = sum(1 for c in letters if c.isupper())
    return caps / len(letters)

def elongation_count(raw_text):
    return len(re.findall(r"(.)\1{2,}", raw_text))

def word_count(raw_text):
    return len(raw_text.split())

def extract_custom_features(raw_text, cleaned_text):
    return np.array([
        negation_count(cleaned_text),
        exclamation_count(raw_text),
        question_count(raw_text),
        caps_ratio(raw_text),
        elongation_count(raw_text),
        word_count(raw_text),
    ], dtype=np.float64)

In [38]:

def predict_sentiment(raw_tweet):
    cleaned = clean_text(raw_tweet)
    processed = process(cleaned)

    tfidf_vec = vectorizer.transform([processed])
    custom_vec = extract_custom_features(raw_tweet, processed).reshape(1, -1)

    combined = hstack([tfidf_vec, csr_matrix(custom_vec)])

    prediction = model.predict(combined)[0]
    return LABEL_MAP.get(prediction, str(prediction))

In [39]:
test_tweets = [
    "I don't hate this movie",
    "Not bad for a first try",
]
test_tweets += [
    "Oh great, my flight got cancelled again. Just perfect.",
    "Wow, another Monday. Can't wait.",
    "Love it when my phone dies right before an important call",
]
test_tweets += [
    "The food was amazing but the service was terrible",
    "Great acting, but the plot was a complete mess",
    "I love the design, hate the price",
]
test_tweets += [
    "I wish this was as good as everyone said",
    "Expected it to be great, turned out disappointing",
]
test_tweets += [
    "meh",
    "it's fine I guess",
    "okay-ish",
]
for tweet in test_tweets:
    sentiment = predict_sentiment(tweet)
    print(f"{sentiment:10s} | {tweet}")

negative   | I don't hate this movie
negative   | Not bad for a first try
negative   | Oh great, my flight got cancelled again. Just perfect.
positive   | Wow, another Monday. Can't wait.
negative   | Love it when my phone dies right before an important call
negative   | The food was amazing but the service was terrible
positive   | Great acting, but the plot was a complete mess
negative   | I love the design, hate the price
negative   | I wish this was as good as everyone said
negative   | Expected it to be great, turned out disappointing
negative   | meh
positive   | it's fine I guess
positive   | okay-ish


In [40]:
tweet = input("Enter a tweet: ")
sentiment = predict_sentiment(tweet)

print(f"\nTweet: {tweet}")
print(f"Predicted sentiment: {sentiment}")

Enter a tweet: Not bad

Tweet: Not bad
Predicted sentiment: negative
